# The five independently trained models

Every matrix in the paper is the per-pair median of five training runs of the binding model, each
on its own stratified 80/10/10 split of the M2OR observations. This notebook shows two things the
median hides: how well each run does on its own held-out test split, and whether the paper's results
hold when the calls come from one run alone.

The per-run predictions are part of the Zenodo prediction archive (`python scripts/download_zenodo_data.py`),
which unpacks them into `Downstream_Analysis/predictions/runs/`.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path("../..").resolve()
sys.path.append(str(REPO / "Supplementary_Material"))
import robustness_checks as rc
from robustness_checks import ac
from plotting_functions import custom_plots, FIGSIZE
from activation_figures import CLASS_I_COLOUR, CLASS_II_COLOUR, INK, MUTED, save

plt.rcParams.update(custom_plots())
plt.rcParams.update(rc.TEXT_SIZES)
pd.set_option("display.max_columns", 50)

TABLES = Path("Tables")
FIGURES = Path("Figures")
TABLES.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

## Performance on held-out test pairs

The model check of Extended Data Fig. 2 compares the median prediction with the 23,782 measured
human pairs. Most of those pairs were in the training data of most runs, so that comparison shows how
well the predictions reproduce the measurements, not how well they generalize. The numbers below are
each run's own test split, pairs the run never saw. The positive rate of M2OR is about 6%, which is
the PR AUC of a model that ranks at random.

In [2]:
test_metrics = pd.read_csv(rc.RUN_METRICS)
per_run = test_metrics[test_metrics.run.str.startswith("Run")].copy()
for column in per_run.columns[1:]:
    per_run[column] = per_run[column].astype(float)
summary = pd.DataFrame([dict(run="mean", **per_run.iloc[:, 1:].mean()),
                        dict(run="s.d.", **per_run.iloc[:, 1:].std(ddof=0))])
held_out = pd.concat([per_run, summary], ignore_index=True)
held_out.to_csv(TABLES / "held_out_test_metrics.csv", index=False)
held_out.round(3)

,run,pr_auc,roc_auc,balanced_accuracy,f1,accuracy,precision,recall,mcc,per_label_accuracy_negative,per_label_accuracy_positive
0,Run_1,0.500,0.920,0.846,0.392,0.846,0.255,0.847,0.411,0.845,0.847
1,Run_2,0.500,0.904,0.830,0.368,0.833,0.237,0.827,0.385,0.833,0.827
2,Run_3,0.504,0.912,0.834,0.378,0.840,0.245,0.827,0.394,0.841,0.827
3,Run_4,0.495,0.922,0.858,0.426,0.865,0.284,0.850,0.443,0.866,0.850
4,Run_5,0.471,0.896,0.813,0.407,0.872,0.280,0.746,0.405,0.880,0.746
5,mean,0.494,0.911,0.836,0.394,0.851,0.260,0.820,0.408,0.853,0.820
6,s.d.,0.012,0.010,0.015,0.021,0.015,0.019,0.038,0.020,0.017,0.038


## The paper's results from each run alone

Each run's reference-tree and ancestral predictions are binarized exactly as the median is in the
paper: a calibration map fitted on that run's scores for the measured human pairs, a density fill of
each matrix, measured values written over the extant matrix for the extant results and not for the
ancestral ones.

In [3]:
from calibration import density_fill, fit_calibration_map

extant = rc.load_extant_inputs()
data = extant.data
ancestral = rc.load_ancestral_inputs()

extant_rows, ancestral_rows = [], []
for run in range(1, 6):
    reference = pd.read_csv(rc.RUNS / f"Reference_Tree_Predictions/reference_tree_predictions_run_{run}.csv")
    reference = reference.pivot(index="protein_id", columns="smiles_id", values="probability")
    scores = reference.loc[data.receptor_ids, data.odorant_ids].values
    run_map = fit_calibration_map(scores[data.measured_rows, data.measured_cols], data.measured_labels)

    hybrid = density_fill(scores, run_map, tested_mask=data.tested, experimental=data.experimental)
    extant_rows.append(dict(run=run, **rc.extant_summary(hybrid.matrix.astype(int), extant)))

    reconstructed = pd.read_csv(rc.RUNS / f"ASR_Predictions/ASR_predictions_run_{run}.csv")
    reconstructed = reconstructed.pivot(index="protein_id", columns="smiles_id", values="probability")
    reconstructed.index = [str(name).replace("_ASR", "") for name in reconstructed.index]
    reconstructed = reconstructed.loc[ancestral.ancestral_scores.index, ancestral.ancestral_scores.columns]
    tips = reference.loc[ancestral.extant_scores.index, ancestral.extant_scores.columns]
    binary = rc.ancestral_calls(density_fill(reconstructed.values, run_map).matrix,
                                density_fill(tips.values, run_map).matrix, ancestral)
    ancestral_rows.append(dict(run=run, **rc.ancestral_summary(binary, ancestral)))

extant_by_run = pd.DataFrame(extant_rows)
ancestral_by_run = pd.DataFrame(ancestral_rows)
extant_by_run.to_csv(TABLES / "extant_by_run.csv", index=False)
ancestral_by_run.to_csv(TABLES / "ancestral_by_run.csv", index=False)
extant_by_run.round(3).set_index("run").T

run,1,2,3,4,5
called_cells,9594,10653,11233,10300,9830
odorants_with_a_call,668,721,675,693,686
median_ligands_class_I,13.5,15.5,11.0,18.0,11.0
median_ligands_class_II,7.0,3.0,4.0,5.0,4.0
breadth_P,0.179,0.001,0.016,0.0,0.008
n_class_I_only,131,147,75,137,129
pct_acid_class_I_only,47.328,44.218,56.0,32.847,34.884
pct_aromatic_class_I_only,24.427,18.367,20.0,22.628,20.93
n_shared,188,332,315,296,294
pct_acid_shared,2.66,3.012,4.762,7.432,6.803


In [4]:
ancestral_by_run.round(4).set_index("run").T

run,1,2,3,4,5
node_0_ligands,25,48,9,50,2
node_1_ligands,10,28,24,46,22
node_2_ligands,22,3,2,14,6
node_5_ligands,15,95,21,36,8
node_0_to_1_kept,0,6,3,9,0
node_0_to_1_lost,25,42,6,41,2
node_0_to_1_gained,10,22,21,37,22
node_0_to_2_kept,16,3,1,14,2
node_0_to_2_lost,9,45,8,36,0
node_0_to_2_gained,6,0,1,0,4


In every run, the odorants only class I reads are acid-rich and those only class II reads carry
almost no acids, the gains along the class I founding branch are enriched for carboxylic acids, the
functional half-life is 0.12-0.18 substitutions per site and ancestors are broader than extant
receptors. The six clusters keep an enriched descriptor each in four of five runs. The results that
rest on a handful of molecules at single nodes are less stable from run to run: the aromatic gain
at the second duplication is significant in three runs, and odorants read by two receptors are present
at the first duplication in four (one to four odorants). The paper's matrices are the median over the
five runs, which is more reliable than any single run; these numbers show how much each result
depends on the training split.